# Tourenkarte für Soll- und Ist-Daten (PRC)

Dieses Notebook liest `Ist-Daten.csv` und `prc_mit_tournummer.csv` ein und erstellt eine interaktive Karte mit:

- Soll- und Ist-Touren
- verbundenen Punkten
- Hover-Infos für Abfahrt und Ankunft
- Stoppreihenfolge als Zahl im Marker
- automatischem Zoom auf die gewählte Tour
- umschaltbaren Tournummern über die Layer-Auswahl

Die PRC-Daten sind dabei Rohdaten mit zugeordneter Tournummer.


In [ ]:
import pandas as pd
import folium
from folium.plugins import BeautifyIcon
from branca.element import Template, MacroElement
from IPython.display import display


In [4]:
# Dateipfade und Ausgabedatei definieren
IST_FILE = 'prc_mit_tournummer.csv'
SOLL_FILE = 'Soll-Daten.csv'
OUTPUT_HTML = 'tourenkarte_prc.html'


def normalize_tour(x):
    # Normiert die Tournummer: None für leere Werte, ansonsten getrimmter String
    if pd.isna(x):
        return None
    s = str(x).strip()
    return s if s else None


def parse_dt(series):
    # Versucht, eine Serie in Datetime-Werte zu konvertieren
    return pd.to_datetime(series, errors='coerce')


def fmt_dt(x):
    # Formatiert Datum und Uhrzeit für Popup- und Tooltip-Anzeigen
    if pd.isna(x):
        return '-'
    return pd.Timestamp(x).strftime('%d.%m.%Y %H:%M')

def clean_ist(df):
    # Bereinigung und Vorbereitung der Ist-Daten
    df = df.copy()
    df.columns = df.columns.str.strip()

    # Tournummern als sauberer String
    df["TOURNR"] = df["TOURNR"].astype(str).str.strip()

    # Koordinaten standardisieren und Kommas als Dezimaltrenner ersetzen
    df["Latitude"] = (
        df["Latitude"]
        .astype(str)
        .str.strip()
        .str.replace(",", ".", regex=False)
    )
    df["Longitude"] = (
        df["Longitude"]
        .astype(str)
        .str.strip()
        .str.replace(",", ".", regex=False)
    )

    df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
    df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")

    # Fahrzeugzeitstempel konvertieren
    df["Zeitstempel_Fahrzeug"] = pd.to_datetime(df["Zeitstempel_Fahrzeug"], errors="coerce")

    # Ungültige Zeilen ohne Tour oder Koordinaten entfernen
    df = df.dropna(subset=["TOURNR", "Latitude", "Longitude"]).copy()

    # Reihenfolge der Stopps pro Tour bestimmen
    df = df.sort_values(["TOURNR", "Zeitstempel_Fahrzeug", "DATUM"]).reset_index(drop=True)
    df["stop_no"] = df.groupby("TOURNR").cumcount() + 1

    return df


def clean_soll(df):
    # Bereinigung und Vorbereitung der Soll-Daten
    df = df.copy()
    df['TOURNR'] = df['TOURNR'].map(normalize_tour)
    df['GEOX'] = pd.to_numeric(df['GEOX'], errors='coerce')
    df['GEOY'] = pd.to_numeric(df['GEOY'], errors='coerce')
    df['ABFAHRT_dt'] = parse_dt(df['ABFAHRT'])
    df['ANKUNFT_dt'] = parse_dt(df['ANKUNFT'])

    # Ungültige Touren und Koordinaten entfernen
    df = df.dropna(subset=['TOURNR', 'GEOY', 'GEOX'])
    # Nach Tour und Zeit sortieren, um die Stoppreihenfolge zu setzen
    df = df.sort_values(['TOURNR', 'ABFAHRT_dt', 'ANKUNFT_dt', 'DATUM']).reset_index(drop=True)
    df['stop_no'] = df.groupby('TOURNR').cumcount() + 1
    return df


In [ ]:
# Daten aus den CSV-Dateien einlesen
df_ist = pd.read_csv(IST_FILE)
df_soll = pd.read_csv(SOLL_FILE)

# Daten säubern und in ein einheitliches Format bringen
df_ist = clean_ist(df_ist)
df_soll = clean_soll(df_soll)

# Grundlegende Informationen zur Datenform ausgeben
print('Ist-Form:', df_ist.shape)
print('Soll-Form:', df_soll.shape)
print('Anzahl Touren in Ist:', df_ist['TOURNR'].nunique())
print('Anzahl Touren in Soll:', df_soll['TOURNR'].nunique())

# Erste Zeilen der aufbereiteten Tabellen anzeigen
display(df_ist.head())
display(df_soll.head())


Ist-Form: (29027, 19)
Soll-Form: (1620, 13)
Anzahl Touren in Ist: 201
Anzahl Touren in Soll: 240


,Unnamed: 0,msg_typ,PositionsID,Longitude,Latitude,Geschwindigkeit,KMStand,Zeitstempel_GPS,Zeitstempel_Fahrzeug,Zeitstempel_Server,LKW_KENNZ,Status,TourID,StationID,SendposID,Ereignis_Typ,DATUM,TOURNR,stop_no
0,22,Position,16054978711,9.98546,53.52335,0,339687,2026-03-01 23:59:59,2026-03-02 00:00:00,2026-03-02 00:00:47,Plo-TS856,NaN,H/42374,NaN,NaN,NaN,2026-03-02,H/42374,1
1,149,Position,16056142197,9.98547,53.52335,0,339687,2026-03-02 05:59:59,2026-03-02 06:00:00,2026-03-02 06:00:49,Plo-TS856,NaN,H/42374,NaN,NaN,NaN,2026-03-02,H/42374,2
2,196,Position,16056260113,9.98639,53.52345,20,339687,2026-03-02 06:16:45,2026-03-02 06:16:45,2026-03-02 06:17:04,Plo-TS856,NaN,H/42374,NaN,NaN,NaN,2026-03-02,H/42374,3
3,197,Position,16056260217,9.98648,53.52344,20,339687,2026-03-02 06:16:46,2026-03-02 06:16:46,2026-03-02 06:17:04,Plo-TS856,NaN,H/42374,NaN,NaN,NaN,2026-03-02,H/42374,4
4,336,Position,16056552191,9.98827,53.52320,20,339688,2026-03-02 06:57:54,2026-03-02 06:57:54,2026-03-02 06:58:21,Plo-TS856,NaN,H/42374,NaN,NaN,NaN,2026-03-02,H/42374,5


,TOURNR,LKW_KENNZ,FAHRERNAME,TOUR,ANKUNFT,ABFAHRT,GEOX,GEOY,STOPP_DAUER,DATUM,ABFAHRT_dt,ANKUNFT_dt,stop_no
0,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 06:00:00,2026-03-02 06:30:00,9.98632,53.52348,0 days 00:30:00,2026-03-02,2026-03-02 06:30:00,2026-03-02 06:00:00,1
1,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 06:49:00,2026-03-02 07:19:00,9.99383,53.54751,0 days 00:30:00,2026-03-02,2026-03-02 07:19:00,2026-03-02 06:49:00,2
2,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 07:41:00,2026-03-02 08:11:00,9.98632,53.52348,0 days 00:30:00,2026-03-02,2026-03-02 08:11:00,2026-03-02 07:41:00,3
3,H/42374,Plo-TS856,"Hesse, Sven-Erik",H/42374,2026-03-02 08:31:00,2026-03-02 09:01:00,9.99383,53.54751,0 days 00:30:00,2026-03-02,2026-03-02 09:01:00,2026-03-02 08:31:00,4
4,H/42375,OD-TS8050,"Szmajdewicz, Marcin",H/42375,2026-03-02 06:00:00,2026-03-02 06:30:00,9.98632,53.52348,0 days 00:30:00,2026-03-02,2026-03-02 06:30:00,2026-03-02 06:00:00,1


In [ ]:
# Alle eindeutigen Tournummern aus Soll- und Ist-Daten sammeln
all_tours = sorted(set(df_ist['TOURNR'].dropna()) | set(df_soll['TOURNR'].dropna()))
if not all_tours:
    raise ValueError('Keine Tournummern gefunden.')

# Standardtour für die initiale Kartendarstellung wählen
default_tour = all_tours[0]
print('Standardtour:', default_tour)

# Die ersten 20 Tournummern anzeigen, um die Daten zu prüfen
print('Erste 20 Tournummern:', all_tours[:20])


Standardtour: H/42374
Erste 20 Tournummern: ['H/42374', 'H/42375', 'H/42376', 'H/42377', 'H/42378', 'H/42379', 'H/42380', 'H/42381', 'H/42383', 'H/42391', 'H/42392', 'H/42393', 'H/42395', 'H/42396', 'H/42397', 'H/42398', 'H/42400', 'H/42403', 'H/42404', 'H/42405']


In [ ]:
# Interaktive Folium-Karte initialisieren
m = folium.Map(
    location=[54.3, 10.1],
    zoom_start=8,
    tiles='CartoDB positron',
    control_scale=True
)

# Bounding-Box-Daten pro Tour für automatisches Zoomen speichern
bounds_by_tour = {}
layer_js_names = []

for tour in all_tours:
    # FeatureGroup pro Tour, damit Layers ein- und ausgeschaltet werden können
    fg = folium.FeatureGroup(name=f'Tour {tour}', show=(tour == default_tour))
    tour_points = []

    soll_t = df_soll[df_soll['TOURNR'] == tour].copy()
    ist_t = df_ist[df_ist['TOURNR'] == tour].copy()

    if not soll_t.empty:
        soll_coords = soll_t[['GEOY', 'GEOX']].values.tolist()
        folium.PolyLine(
            locations=soll_coords,
            color='blue',
            weight=4,
            opacity=0.85,
            tooltip=f'Soll-Tour {tour}',
        ).add_to(fg)

        for _, row in soll_t.iterrows():
            lat, lon = row['GEOY'], row['GEOX']
            tour_points.append([lat, lon])

            tooltip_html = f"""
            <div style=\"font-size:13px;\">
                <b>Soll</b><br>
                Tour: {tour}<br>
                Stopp: {int(row['stop_no'])}<br>
                Abfahrt: {fmt_dt(row['ABFAHRT_dt'])}<br>
                Ankunft: {fmt_dt(row['ANKUNFT_dt'])}
            </div>
            """

            folium.Marker(
                [lat, lon],
                tooltip=folium.Tooltip(tooltip_html, sticky=True),
                popup=folium.Popup(tooltip_html, max_width=260),
                icon=BeautifyIcon(
                    icon_shape='marker',
                    number=int(row['stop_no']),
                    border_color='#1d4ed8',
                    text_color='#1d4ed8',
                    background_color='#dbeafe'
                )
            ).add_to(fg)

    if not ist_t.empty:
        ist_coords = ist_t[['Latitude', 'Longitude']].values.tolist()
        folium.PolyLine(
            locations=ist_coords,
            color='red',
            weight=4,
            opacity=0.85,
            dash_array='8,6',
            tooltip=f'Ist-Tour {tour}',
        ).add_to(fg)

        for _, row in ist_t.iterrows():
            lat, lon = row['Latitude'], row['Longitude']
            tour_points.append([lat, lon])

            tooltip_html = f"""
            <div style=\"font-size:13px;\">
                <b>Ist</b><br>
                Tour: {tour}<br>
                Stopp: {int(row['stop_no'])}<br>
                Zeitstempel Fahrtzeug: {fmt_dt(row['Zeitstempel_Fahrzeug'])}<br>
            </div>
            """

            folium.Marker(
                [lat, lon],
                tooltip=folium.Tooltip(tooltip_html, sticky=True),
                popup=folium.Popup(tooltip_html, max_width=260),
                icon=BeautifyIcon(
                    icon_shape='marker',
                    number=int(row['stop_no']),
                    border_color='#b91c1c',
                    text_color='#b91c1c',
                    background_color='#fee2e2'
                )
            ).add_to(fg)

    if tour_points:
        # Bounding Box der Tour berechnen, um später auf diese Tour zu zoomen
        lats = [p[0] for p in tour_points]
        lons = [p[1] for p in tour_points]
        bounds_by_tour[tour] = [[min(lats), min(lons)], [max(lats), max(lons)]]

    fg.add_to(m)
    layer_js_names.append((f'Tour {tour}', fg.get_name(), tour))

# Layer-Steuerung für die Tourauswahl hinzufügen
folium.LayerControl(collapsed=False).add_to(m)

# Legende für Soll- und Ist-Touren einfügen
legend_html = """
<div style="
 position: fixed;
 bottom: 20px;
 left: 20px;
 z-index: 9999;
 background: white;
 border: 1px solid #ccc;
 border-radius: 8px;
 padding: 10px 12px;
 font-size: 13px;
 box-shadow: 0 2px 8px rgba(0,0,0,0.15);
">
  <div style="font-weight:600; margin-bottom:6px;">Legende</div>
  <div><span style="display:inline-block;width:14px;height:4px;background:blue;margin-right:6px;vertical-align:middle;"></span> Soll</div>
  <div><span style="display:inline-block;width:14px;height:4px;background:red;margin-right:6px;vertical-align:middle;"></span> Ist</div>
  <div style="margin-top:6px;color:#555;">Zahl = Reihenfolge des Stopps</div>
</div>
"""

m.get_root().html.add_child(folium.Element(legend_html))

# JavaScript für automatisches Zoomen bei Layer-Auswahl vorbereiten
map_var = m.get_name()
layer_map_js = ',\n'.join([f'\"{label}\": {js_name}' for label, js_name, _ in layer_js_names])
bounds_js = ',\n'.join([f'\"{tour}\": {bounds}' for tour, bounds in bounds_by_tour.items()])

script = f"""
{{% macro script(this, kwargs) %}}
var layerMap = {{
{layer_map_js}
}};

var tourBounds = {{
{bounds_js}
}};

var mapObj = {map_var};

function fitTourByLayerName(layerName) {{
    var tour = layerName.replace('Tour ', '');
    var b = tourBounds[tour];
    if (b) {{
        mapObj.fitBounds(b, {{
            padding: [40, 40],
            maxZoom: 15
        }});
    }}
}}

mapObj.whenReady(function() {{
    fitTourByLayerName('Tour {default_tour}');
}});

mapObj.on('overlayadd', function(e) {{
    for (const [name, layer] of Object.entries(layerMap)) {{
        if (e.layer === layer) {{
            fitTourByLayerName(name);
            break;
        }}
    }}
}});
{{% endmacro %}}
"""

macro = MacroElement()
macro._template = Template(script)
m.get_root().add_child(macro)

# Karte als HTML-Datei speichern und ausgeben
m.save(OUTPUT_HTML)
print(f'Karte gespeichert als: {OUTPUT_HTML}')

m


Karte gespeichert als: tourenkarte_prc.html
